In [1]:
%pip install -q xgboost scikit-learn pandas numpy shap

Note: you may need to restart the kernel to use updated packages.


In [2]:
import warnings
import numpy as np
import pandas as pd
import shap

from xgboost import XGBClassifier
from sklearn.ensemble import RandomForestClassifier, IsolationForest
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.metrics import (
    classification_report,
    accuracy_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
)

warnings.filterwarnings("ignore")


/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
EXPECTED_COLUMNS = [
    "Container_ID", "Declaration_Date", "Declaration_Time", "Trade_Regime",
    "Origin_Country", "Destination_Port", "Destination_Country", "HS_Code",
    "Importer_ID", "Exporter_ID", "Declared_Value", "Declared_Weight",
    "Measured_Weight", "Shipping_Line", "Dwell_Time_Hours", "Clearance_Status",
]

# Column name aliases — maps raw CSV header variants to canonical names
COLUMN_ALIASES = {
    "Declaration_Date (YYYY-MM-DD)": "Declaration_Date",
    "Trade_Regime (Import / Export / Transit)": "Trade_Regime",
}


def load_data(filepath: str) -> pd.DataFrame:
    """Read shipment CSV, normalise column names, and validate required columns."""
    df = pd.read_csv(filepath, dtype={"HS_Code": str})
    # Strip surrounding whitespace from headers
    df.columns = df.columns.str.strip()
    # Rename aliased headers to canonical names
    df.rename(columns=COLUMN_ALIASES, inplace=True)
    missing = [c for c in EXPECTED_COLUMNS if c not in df.columns]
    if missing:
        raise ValueError(f"Missing expected columns: {missing}")
    return df


df_raw = load_data("Historical Data.csv")
print(f"Loaded {len(df_raw):,} rows | columns: {list(df_raw.columns)}")

Loaded 54,000 rows | columns: ['Container_ID', 'Declaration_Date', 'Declaration_Time', 'Trade_Regime', 'Origin_Country', 'Destination_Port', 'Destination_Country', 'HS_Code', 'Importer_ID', 'Exporter_ID', 'Declared_Value', 'Declared_Weight', 'Measured_Weight', 'Shipping_Line', 'Dwell_Time_Hours', 'Clearance_Status']


In [4]:
# Clearance statuses that indicate a risky / flagged shipment
RISK_STATUSES = {"critical", "low risk", "hold", "inspect", "detained", "flagged"}


def preprocess_data(df: pd.DataFrame) -> pd.DataFrame:
    """
    Clean and prepare the raw dataframe.
    Steps:
      - Parse dates and extract temporal features (day, month, day-of-week)
      - Impute missing values
      - Clip extreme numeric outliers
      - Derive binary Risk_Label from Clearance_Status
    """
    df = df.copy()

    # --- Temporal parsing ---
    df["Declaration_Date"] = pd.to_datetime(df["Declaration_Date"], errors="coerce")
    df["declaration_day"]   = df["Declaration_Date"].dt.day          # 1–31
    df["declaration_month"] = df["Declaration_Date"].dt.month        # 1–12
    df["declaration_dow"]   = df["Declaration_Date"].dt.dayofweek    # 0=Mon … 6=Sun

    # --- Missing value imputation ---
    num_cols = ["Declared_Value", "Declared_Weight", "Measured_Weight", "Dwell_Time_Hours"]
    for col in num_cols:
        df[col] = pd.to_numeric(df[col], errors="coerce")
        df[col].fillna(df[col].median(), inplace=True)

    cat_cols = ["Trade_Regime", "Origin_Country", "Destination_Country",
                "Destination_Port", "Shipping_Line"]
    for col in cat_cols:
        df[col].fillna(df[col].mode()[0], inplace=True)

    # --- Outlier clipping (99th percentile cap) ---
    for col in ["Declared_Value", "Declared_Weight"]:
        cap = df[col].quantile(0.99)
        df[col] = df[col].clip(upper=cap)

    # --- Binary target: 1 = risky, 0 = Clear ---
    df["Risk_Label"] = df["Clearance_Status"].str.strip().str.lower().apply(
        lambda s: "RISKY" if s in RISK_STATUSES else "CLEAR"
    )

    return df


df_clean = preprocess_data(df_raw)
print(f"Class distribution:\n{df_clean['Risk_Label'].value_counts().to_string()}")


Class distribution:
Risk_Label
CLEAR    42347
RISKY    11653


In [5]:
HIGH_RISK_HS_PREFIXES = {
    "93", "28", "29", "30", "36", "38", "85", "62", "64",
}

HIGH_RISK_ROUTES = {
    ("CN", "EU"), ("VN", "US"), ("TR", "EU"), ("PK", "GB"),
    ("NG", "GB"), ("BD", "US"), ("MX", "US"),
}

IF_FEATURE_COLS = [
    "Declared_Value", "Declared_Weight", "Measured_Weight",
    "Weight_Diff", "Weight_Deviation_Ratio", "Value_Per_Weight",
    "Dwell_Time_Hours", "Weight_vs_HS_Avg",
]


def _base_features(df: pd.DataFrame) -> pd.DataFrame:
    """
    Compute all features that do NOT require cross-row group statistics.
    Safe to apply independently on any split.
    """
    df = df.copy()
    EPS = 1e-5

    df["Weight_Diff"]            = (df["Declared_Weight"] - df["Measured_Weight"]).abs()
    df["Weight_Deviation_Ratio"] = df["Weight_Diff"] / (df["Declared_Weight"] + EPS)
    df["Value_Per_Weight"]       = df["Declared_Value"] / (df["Declared_Weight"] + EPS)
    df["Weight_Direction"]       = df["Measured_Weight"] - df["Declared_Weight"]

    df["hs_code_risk_flag"] = df["HS_Code"].astype(str).str[:2].isin(HIGH_RISK_HS_PREFIXES).astype(int)
    df["route_risk_flag"]   = df.apply(
        lambda r: int((r["Origin_Country"], r["Destination_Country"]) in HIGH_RISK_ROUTES), axis=1
    )

    dwell_95 = df["Dwell_Time_Hours"].quantile(0.95)
    df["dwell_time_anomaly"] = (df["Dwell_Time_Hours"] > dwell_95).astype(int)

    df["declaration_hour"] = pd.to_datetime(
        df["Declaration_Time"], format="%H:%M:%S", errors="coerce"
    ).dt.hour.fillna(0).astype(int)
    df["is_weekend"] = (df["declaration_dow"] >= 5).astype(int)

    return df


def _fit_group_stats(df_train: pd.DataFrame) -> dict:
    """
    Compute group-level aggregation statistics on the TRAINING set only.
    Returns a lookup dict used to map stats onto any split without leakage.
    """
    EPS = 1e-5
    hs_avg   = df_train.groupby("HS_Code")["Declared_Weight"].mean().rename("hs_avg_weight")
    exp_avg  = df_train.groupby(["Exporter_ID", "HS_Code"])["Declared_Weight"].mean().rename("exp_hs_avg")
    imp_freq = df_train.groupby("Importer_ID")["Importer_ID"].count().rename("importer_frequency")
    exp_freq = df_train.groupby("Exporter_ID")["Exporter_ID"].count().rename("exporter_frequency")

    # Global fallback values (training medians) for unseen groups in test/inference
    global_hs_avg  = df_train["Declared_Weight"].mean()
    global_exp_avg = df_train["Declared_Weight"].mean()

    return dict(
        hs_avg=hs_avg,
        exp_avg=exp_avg,
        imp_freq=imp_freq,
        exp_freq=exp_freq,
        global_hs_avg=global_hs_avg,
        global_exp_avg=global_exp_avg,
    )


def _apply_group_stats(df: pd.DataFrame, stats: dict) -> pd.DataFrame:
    """
    Map pre-computed training group stats onto any dataframe split.
    Unknown groups (unseen in training) receive global training fallback values.
    """
    df = df.copy()
    EPS = 1e-5

    # HS-code weight average — fallback to global train mean for unseen HS codes
    df["Expected_Weight_By_HS"] = df["HS_Code"].map(stats["hs_avg"]).fillna(stats["global_hs_avg"])
    df["Weight_vs_HS_Avg"]      = df["Declared_Weight"] / (df["Expected_Weight_By_HS"] + EPS)

    # Exporter × HS-code average — fallback to global train mean
    exp_hs_key = list(zip(df["Exporter_ID"], df["HS_Code"]))
    df["Exporter_Avg_Weight_By_HS"] = [
        stats["exp_avg"].get(k, stats["global_exp_avg"]) for k in exp_hs_key
    ]
    df["Exporter_Weight_Deviation"] = df["Declared_Weight"] - df["Exporter_Avg_Weight_By_HS"]

    # Entity frequency — fallback to 1 (lowest known frequency) for unseen entities
    df["importer_frequency"] = df["Importer_ID"].map(stats["imp_freq"]).fillna(1).astype(int)
    df["exporter_frequency"] = df["Exporter_ID"].map(stats["exp_freq"]).fillna(1).astype(int)

    return df


def engineer_features(df: pd.DataFrame, stats: dict | None = None) -> pd.DataFrame:
    """
    Full feature engineering.
    - If stats=None  → compute group stats from df itself (training path).
    - If stats=dict  → apply pre-fitted stats from training (test/inference path, no leakage).
    Returns (df_engineered, stats_dict).
    """
    df = _base_features(df)
    if stats is None:
        stats = _fit_group_stats(df)
    df = _apply_group_stats(df, stats)
    return df, stats


# Fit on the full labelled dataset; stats will be recomputed properly
# after the train/test split inside the model training cell.
df_feat, _ = engineer_features(df_clean)
print(f"Feature engineering complete | shape: {df_feat.shape}")


Feature engineering complete | shape: (54000, 35)


In [6]:
CATEGORICAL_COLS = [
    "Trade_Regime", "Origin_Country", "Destination_Country",
    "Destination_Port", "Shipping_Line",
]

DROP_COLS = [
    "Container_ID", "Declaration_Date", "Declaration_Time",
    "Clearance_Status", "Importer_ID", "Exporter_ID", "HS_Code",
    "Risk_Label",
    "Expected_Weight_By_HS", "Exporter_Avg_Weight_By_HS",
]


def encode_features(df: pd.DataFrame, fit_encoders: bool = True, encoders: dict | None = None):
    """
    Label-encode categorical columns.
    - fit_encoders=True  → fit new encoders (training path).
    - fit_encoders=False → apply pre-fitted encoders (test/inference path).
    Returns (X, y, encoders_dict).  y is None when Risk_Label is absent.
    """
    df = df.copy()
    if encoders is None:
        encoders = {}

    for col in CATEGORICAL_COLS:
        if fit_encoders:
            le = LabelEncoder()
            df[col] = le.fit_transform(df[col].astype(str))
            encoders[col] = le
        else:
            le = encoders[col]
            known = set(le.classes_)
            df[col] = df[col].astype(str).apply(lambda v: v if v in known else le.classes_[0])
            df[col] = le.transform(df[col])

    feature_cols = [c for c in df.columns if c not in DROP_COLS]
    X = df[feature_cols]
    y = df["Risk_Label"].map({"RISKY": 1, "CLEAR": 0}) if "Risk_Label" in df.columns else None

    return X, y, encoders


X, y, encoders = encode_features(df_feat)
print(f"Feature matrix shape : {X.shape}")
print(f"Features             : {list(X.columns)}")
print(f"Target distribution  : {y.value_counts().to_dict()}")


Feature matrix shape : (54000, 25)
Features             : ['Trade_Regime', 'Origin_Country', 'Destination_Port', 'Destination_Country', 'Declared_Value', 'Declared_Weight', 'Measured_Weight', 'Shipping_Line', 'Dwell_Time_Hours', 'declaration_day', 'declaration_month', 'declaration_dow', 'Weight_Diff', 'Weight_Deviation_Ratio', 'Value_Per_Weight', 'Weight_Direction', 'hs_code_risk_flag', 'route_risk_flag', 'dwell_time_anomaly', 'declaration_hour', 'is_weekend', 'Weight_vs_HS_Avg', 'Exporter_Weight_Deviation', 'importer_frequency', 'exporter_frequency']
Target distribution  : {0: 42347, 1: 11653}


In [7]:
# ---------------------------------------------------------------------------
# Step 0 — Split BEFORE feature engineering to prevent data leakage.
# Group-aggregation stats (HS-avg weight, exporter deviation, entity frequency)
# are computed ONLY on the training rows, then mapped onto test rows.
# ---------------------------------------------------------------------------
from sklearn.model_selection import train_test_split as _tts

# Split the cleaned (pre-feature-engineering) dataframe
df_train_raw, df_test_raw = _tts(df_clean, test_size=0.2, stratify=df_clean["Risk_Label"], random_state=42)
df_train_raw = df_train_raw.reset_index(drop=True)
df_test_raw  = df_test_raw.reset_index(drop=True)

# Engineer features: fit group stats on train, apply same stats to test
df_train_feat, group_stats = engineer_features(df_train_raw, stats=None)   # fits stats
df_test_feat,  _           = engineer_features(df_test_raw,  stats=group_stats)  # uses train stats

# Encode: fit label encoders on train, transform test with the same encoders
X_train, y_train, encoders = encode_features(df_train_feat, fit_encoders=True)
X_test,  y_test,  _        = encode_features(df_test_feat,  fit_encoders=False, encoders=encoders)

# Re-encode full dataset using train-fitted encoders (used for full-dataset predictions later)
X, y, _ = encode_features(df_feat, fit_encoders=False, encoders=encoders)

print(f"Train : {X_train.shape}  |  Test : {X_test.shape}")
print(f"Train target dist : {y_train.value_counts().to_dict()}")
print(f"Test  target dist : {y_test.value_counts().to_dict()}")

# ---------------------------------------------------------------------------
# Stage 1 — Isolation Forest fitted on TRAIN IF-features only
# ---------------------------------------------------------------------------
contamination_rate = float(np.clip(y_train.mean(), 0.01, 0.49))

isolation_forest = IsolationForest(
    n_estimators=300,
    contamination=contamination_rate,
    max_samples="auto",
    random_state=42,
    n_jobs=-1,
)
isolation_forest.fit(X_train[IF_FEATURE_COLS])

def _if_augment(X_split: pd.DataFrame) -> pd.DataFrame:
    """Augment a split with normalised Anomaly_Score and Anomaly_Flag."""
    raw   = -isolation_forest.decision_function(X_split[IF_FEATURE_COLS])
    # Normalise using train-set min/max to avoid test leakage
    train_raw = -isolation_forest.decision_function(X_train[IF_FEATURE_COLS])
    lo, hi    = train_raw.min(), train_raw.max()
    normed    = (raw - lo) / (hi - lo + 1e-9)
    out = X_split.copy()
    out["Anomaly_Score"] = np.clip(normed, 0, 1)
    out["Anomaly_Flag"]  = (isolation_forest.predict(X_split[IF_FEATURE_COLS]) == -1).astype(int)
    return out

X_train_aug = _if_augment(X_train)
X_test_aug  = _if_augment(X_test)

# Full dataset augmented (for bulk prediction after training)
X_aug_full = _if_augment(X)

print(f"\nIsolation Forest | train anomaly rate: {X_train_aug['Anomaly_Flag'].mean():.2%}  "
      f"(target ≈ {contamination_rate:.2%})")

# ---------------------------------------------------------------------------
# Stage 2 — XGBoost trained on train split only
# ---------------------------------------------------------------------------
n_neg = (y_train == 0).sum()
n_pos = (y_train == 1).sum()
scale_pos_weight = n_neg / n_pos

xgb_model = XGBClassifier(
    n_estimators=500,
    max_depth=6,
    learning_rate=0.04,
    subsample=0.85,
    colsample_bytree=0.8,
    min_child_weight=3,
    gamma=0.1,
    reg_alpha=0.1,
    reg_lambda=1.0,
    scale_pos_weight=scale_pos_weight,
    eval_metric="logloss",
    random_state=42,
    n_jobs=-1,
)
xgb_model.fit(X_train_aug, y_train)

# ---------------------------------------------------------------------------
# Stage 3 — Ensemble score on the held-out test set
# ---------------------------------------------------------------------------
XGB_WEIGHT     = 0.75
ANOMALY_WEIGHT = 0.25
RISK_THRESHOLD = 0.5

xgb_prob_test      = xgb_model.predict_proba(X_test_aug)[:, 1]
anomaly_score_test = X_test_aug["Anomaly_Score"].values
ens_prob_test      = XGB_WEIGHT * xgb_prob_test + ANOMALY_WEIGHT * anomaly_score_test
ens_pred_test      = (ens_prob_test >= RISK_THRESHOLD).astype(int)

# ---------------------------------------------------------------------------
# Stage 4 — Evaluation metrics
# ---------------------------------------------------------------------------
def print_metrics(name: str, y_true, y_pred, y_prob) -> None:
    acc = accuracy_score(y_true, y_pred)
    f1  = f1_score(y_true, y_pred, average="weighted")
    auc = roc_auc_score(y_true, y_prob)
    cm  = confusion_matrix(y_true, y_pred)
    print(f"\n{'='*58}")
    print(f"  {name}")
    print(f"{'='*58}")
    print(f"  Accuracy : {acc:.4f}  ({acc*100:.2f}%)")
    print(f"  F1-Score : {f1:.4f}  (weighted)")
    print(f"  ROC-AUC  : {auc:.4f}")
    print(f"\n{classification_report(y_true, y_pred, target_names=['Clear', 'Risky'])}")
    print(f"  Confusion Matrix:\n    TN={cm[0,0]}  FP={cm[0,1]}\n    FN={cm[1,0]}  TP={cm[1,1]}")

print_metrics("XGBoost (no leakage)", y_test, xgb_model.predict(X_test_aug), xgb_prob_test)
print_metrics("Ensemble [XGB×0.75 + Anomaly×0.25]", y_test, ens_pred_test, ens_prob_test)

# 5-fold cross-validated AUC — also leak-free because CV is on the cleaned df
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

def _cv_pipeline(train_idx, val_idx):
    """Single CV fold: fit group stats on fold train, transform fold val."""
    fold_train = df_clean.iloc[train_idx].reset_index(drop=True)
    fold_val   = df_clean.iloc[val_idx].reset_index(drop=True)

    ft, gs = engineer_features(fold_train, stats=None)
    fv, _  = engineer_features(fold_val,   stats=gs)

    Xt, yt, enc = encode_features(ft, fit_encoders=True)
    Xv, yv, _   = encode_features(fv, fit_encoders=False, encoders=enc)

    spw = (yt == 0).sum() / max((yt == 1).sum(), 1)
    m = XGBClassifier(n_estimators=500, max_depth=6, learning_rate=0.04,
                      subsample=0.85, colsample_bytree=0.8, min_child_weight=3,
                      scale_pos_weight=spw, eval_metric="logloss",
                      random_state=42, n_jobs=-1)

    if_tmp = IsolationForest(n_estimators=300, contamination=float(np.clip(yt.mean(), 0.01, 0.49)),
                             random_state=42, n_jobs=-1)
    if_tmp.fit(Xt[IF_FEATURE_COLS])

    def aug(X_s):
        raw   = -if_tmp.decision_function(X_s[IF_FEATURE_COLS])
        t_raw = -if_tmp.decision_function(Xt[IF_FEATURE_COLS])
        lo, hi = t_raw.min(), t_raw.max()
        out = X_s.copy()
        out["Anomaly_Score"] = np.clip((raw - lo) / (hi - lo + 1e-9), 0, 1)
        out["Anomaly_Flag"]  = (if_tmp.predict(X_s[IF_FEATURE_COLS]) == -1).astype(int)
        return out

    m.fit(aug(Xt), yt)
    return roc_auc_score(yv, m.predict_proba(aug(Xv))[:, 1])

cv_scores = [_cv_pipeline(ti, vi) for ti, vi in cv.split(df_clean, df_clean["Risk_Label"])]
print(f"\n5-Fold CV ROC-AUC (leak-free): {np.mean(cv_scores):.4f} ± {np.std(cv_scores):.4f}")


Train : (43200, 25)  |  Test : (10800, 25)
Train target dist : {0: 33878, 1: 9322}
Test  target dist : {0: 8469, 1: 2331}

Isolation Forest | train anomaly rate: 21.58%  (target ≈ 21.58%)

  XGBoost (no leakage)
  Accuracy : 0.9985  (99.85%)
  F1-Score : 0.9985  (weighted)
  ROC-AUC  : 1.0000

              precision    recall  f1-score   support

       Clear       1.00      1.00      1.00      8469
       Risky       1.00      1.00      1.00      2331

    accuracy                           1.00     10800
   macro avg       1.00      1.00      1.00     10800
weighted avg       1.00      1.00      1.00     10800

  Confusion Matrix:
    TN=8463  FP=6
    FN=10  TP=2321

  Ensemble [XGB×0.75 + Anomaly×0.25]
  Accuracy : 0.9985  (99.85%)
  F1-Score : 0.9985  (weighted)
  ROC-AUC  : 1.0000

              precision    recall  f1-score   support

       Clear       1.00      1.00      1.00      8469
       Risky       1.00      1.00      1.00      2331

    accuracy                        

In [8]:
XGB_WEIGHT     = 0.75
ANOMALY_WEIGHT = 0.25

CRITICAL_THRESHOLD = 0.65
MEDIUM_THRESHOLD   = 0.40


def _assign_risk_level(score: float) -> str:
    if score >= CRITICAL_THRESHOLD:
        return "Critical"
    if score >= MEDIUM_THRESHOLD:
        return "Medium Risk"
    return "Low Risk"


def predict_risk(
    xgb_model: XGBClassifier,
    X_aug: pd.DataFrame,
    container_ids: pd.Series,
) -> pd.DataFrame:
    """
    Ensemble inference on an already-augmented feature matrix (IF signals included).
    Final_Risk_Score = 0.75 × XGBoost_prob + 0.25 × Anomaly_Score
    """
    xgb_prob      = xgb_model.predict_proba(X_aug)[:, 1]
    anomaly_score = X_aug["Anomaly_Score"].values
    ensemble_score = XGB_WEIGHT * xgb_prob + ANOMALY_WEIGHT * anomaly_score

    return pd.DataFrame({
        "Container_ID": container_ids.values,
        "Risk_Score":   np.round(ensemble_score, 4),
        "Risk_Level":   [_assign_risk_level(s) for s in ensemble_score],
    })


# Full-dataset predictions using the leak-free augmented matrix
results_df = predict_risk(xgb_model, X_aug_full, df_feat["Container_ID"])
print(f"Risk level distribution:\n{results_df['Risk_Level'].value_counts().to_string()}")


Risk level distribution:
Risk_Level
Low Risk       42349
Critical       11642
Medium Risk        9


In [9]:
EXPL_WEIGHT_DEV_RATIO  = 0.20
EXPL_HS_AVG_RATIO_HIGH = 2.00
EXPL_HS_AVG_RATIO_LOW  = 0.50
EXPL_EXPORTER_DEV_PCT  = 0.30
EXPL_VALUE_WEIGHT_Z    = 2.50


def _explain_single(row: pd.Series, vw_mean: float, vw_std: float, dwell_95: float) -> str:
    signals = []

    # Rule 1: declared vs measured weight mismatch
    if row.get("Weight_Deviation_Ratio", 0) > EXPL_WEIGHT_DEV_RATIO:
        direction = "over" if row.get("Weight_Direction", 0) > 0 else "under"
        pct = row["Weight_Deviation_Ratio"] * 100
        signals.append(
            f"Large mismatch between declared and measured weight "
            f"({pct:.1f}% {direction}-declaration detected)."
        )

    # Rule 2: weight vs HS-code peer average
    whs = row.get("Weight_vs_HS_Avg", 1.0)
    if whs > EXPL_HS_AVG_RATIO_HIGH:
        signals.append(
            f"Weight significantly exceeds typical shipments of this HS code "
            f"({whs:.1f}× the commodity average)."
        )
    elif 0 < whs < EXPL_HS_AVG_RATIO_LOW:
        signals.append(
            f"Weight significantly below typical shipments of this HS code "
            f"({whs:.2f}× the commodity average), suggesting possible undervaluation."
        )

    # Rule 3: exporter behavioral deviation
    exp_avg = row.get("Exporter_Avg_Weight_By_HS", None)
    exp_dev = row.get("Exporter_Weight_Deviation", 0)
    if exp_avg and abs(exp_avg) > 1e-3:
        exp_dev_pct = abs(exp_dev) / (abs(exp_avg) + 1e-9)
        if exp_dev_pct > EXPL_EXPORTER_DEV_PCT:
            direction = "higher" if exp_dev > 0 else "lower"
            signals.append(
                f"Shipment weight is {exp_dev_pct*100:.1f}% {direction} than this "
                f"exporter's typical weight for the same HS code."
            )

    # Rule 4: Isolation Forest anomaly flag
    if row.get("Anomaly_Flag", 0) == 1:
        signals.append("Shipment behavior detected as anomalous by unsupervised outlier model.")

    # Rule 5: unusual value-to-weight ratio
    vw = row.get("Value_Per_Weight", None)
    if vw is not None and vw_std > 0:
        z = (vw - vw_mean) / vw_std
        if abs(z) > EXPL_VALUE_WEIGHT_Z:
            tag = "unusually high" if z > 0 else "unusually low"
            signals.append(f"Unusual value-to-weight ratio ({tag}), indicating possible misdeclaration.")

    # Rule 6: dwell time
    if row.get("Dwell_Time_Hours", 0) > dwell_95:
        signals.append("Abnormally long dwell time in port, warranting further scrutiny.")

    # Rule 7: HS code / route risk flags
    if row.get("hs_code_risk_flag", 0) == 1:
        signals.append("Commodity falls under a high-risk HS code category.")
    if row.get("route_risk_flag", 0) == 1:
        signals.append("Shipment traverses a flagged high-risk trade corridor.")

    risk_level = row.get("Risk_Level", "")
    if not signals:
        if risk_level == "Critical":
            return "Container flagged by the ensemble risk model as high-risk; manual inspection recommended."
        if risk_level == "Medium Risk":
            return "Shipment presents moderate risk signals; enhanced monitoring recommended."
        return "No major risk signals detected; routine clearance applicable."

    max_sentences = 2 if risk_level != "Low Risk" else 1
    return " ".join(signals[:max_sentences])


def generate_explanations(
    df_engineered: pd.DataFrame,
    X_aug: pd.DataFrame,
    results_df: pd.DataFrame,
) -> pd.DataFrame:
    """
    Domain-rule explanation generator.
    df_engineered : feature-engineered dataframe (contains weight/exporter cols).
    X_aug         : IF-augmented feature matrix (contains Anomaly_Flag).
    results_df    : output of predict_risk (Container_ID, Risk_Score, Risk_Level).
    """
    results_df = results_df.copy()

    vw_mean  = df_engineered["Value_Per_Weight"].mean()
    vw_std   = df_engineered["Value_Per_Weight"].std()
    dwell_95 = df_engineered["Dwell_Time_Hours"].quantile(0.95)

    merged = results_df.merge(
        df_engineered[["Container_ID", "Weight_Deviation_Ratio", "Weight_Direction",
                       "Weight_vs_HS_Avg", "Exporter_Avg_Weight_By_HS",
                       "Exporter_Weight_Deviation", "Value_Per_Weight",
                       "Dwell_Time_Hours", "hs_code_risk_flag", "route_risk_flag"]],
        on="Container_ID", how="left",
    )

    # Attach Anomaly_Flag from the augmented matrix
    flag_df = pd.DataFrame({
        "Container_ID": df_engineered["Container_ID"].values,
        "Anomaly_Flag": X_aug["Anomaly_Flag"].values,
    })
    merged = merged.merge(flag_df, on="Container_ID", how="left")

    results_df["Explanation_Summary"] = merged.apply(
        lambda row: _explain_single(row, vw_mean, vw_std, dwell_95), axis=1
    ).values
    return results_df


results_df = generate_explanations(df_feat, X_aug_full, results_df)
print("Explanation generation complete.")
print(f"\n{results_df[['Container_ID','Risk_Level','Explanation_Summary']].head(5).to_string(index=False)}")


Explanation generation complete.

 Container_ID Risk_Level                                                                                                             Explanation_Summary
     97061800   Low Risk Weight significantly below typical shipments of this HS code (0.01× the commodity average), suggesting possible undervaluation.
     85945189   Low Risk Weight significantly below typical shipments of this HS code (0.31× the commodity average), suggesting possible undervaluation.
     77854751   Low Risk Weight significantly below typical shipments of this HS code (0.19× the commodity average), suggesting possible undervaluation.
     46925060   Low Risk                                                                   No major risk signals detected; routine clearance applicable.
     34131149   Low Risk Weight significantly below typical shipments of this HS code (0.33× the commodity average), suggesting possible undervaluation.


In [10]:
OUTPUT_COLUMNS = ["Container_ID", "Risk_Score", "Risk_Level", "Explanation_Summary"]


def export_results(results_df: pd.DataFrame, output_path: str = "risk_predictions.csv") -> None:
    """Save the final prediction DataFrame to a CSV file."""
    final_df = results_df[OUTPUT_COLUMNS]
    final_df.to_csv(output_path, index=False)
    print(f"Exported {len(final_df):,} records → {output_path}")


export_results(results_df)

Exported 54,000 records → risk_predictions.csv


In [11]:
def run_inference_pipeline(
    filepath: str,
    xgb_model: XGBClassifier,
    isolation_forest: IsolationForest,
    encoders: dict,
    group_stats: dict,
    output_path: str = "risk_predictions_realtime.csv",
) -> pd.DataFrame:
    """
    Leak-free end-to-end inference on unseen shipment CSV.

    Steps:
      1. Load & preprocess (no label extraction — Clearance_Status may be absent).
      2. Apply pre-fitted group stats from training (no new group averages computed).
      3. Apply pre-fitted LabelEncoders (unseen labels fall back to first known class).
      4. Augment with Isolation Forest anomaly signals (normalised against train range).
      5. Ensemble predict: XGB (75%) + Anomaly_Score (25%).
      6. Generate domain-rule-based explanations.
      7. Export CSV.
    """
    df_new = load_data(filepath)
    df_new = preprocess_data(df_new)

    # Apply train group stats — no re-fitting on new data
    df_new, _ = engineer_features(df_new, stats=group_stats)

    X_new, _, _ = encode_features(df_new, fit_encoders=False, encoders=encoders)

    # Augment using the same fitted isolation_forest, normalised to train range
    train_raw  = -isolation_forest.decision_function(X_train[IF_FEATURE_COLS])
    lo, hi     = train_raw.min(), train_raw.max()
    raw_new    = -isolation_forest.decision_function(X_new[IF_FEATURE_COLS])
    X_new_aug  = X_new.copy()
    X_new_aug["Anomaly_Score"] = np.clip((raw_new - lo) / (hi - lo + 1e-9), 0, 1)
    X_new_aug["Anomaly_Flag"]  = (isolation_forest.predict(X_new[IF_FEATURE_COLS]) == -1).astype(int)

    results_new = predict_risk(xgb_model, X_new_aug, df_new["Container_ID"])
    results_new = generate_explanations(df_new, X_new_aug, results_new)
    export_results(results_new, output_path)
    return results_new


df_realtime = run_inference_pipeline(
    filepath="Real-Time Data.csv",
    xgb_model=xgb_model,
    isolation_forest=isolation_forest,
    encoders=encoders,
    group_stats=group_stats,
    output_path="risk_predictions_realtime.csv",
)

df_realtime.head(10)


Exported 8,481 records → risk_predictions_realtime.csv


,Container_ID,Risk_Score,Risk_Level,Explanation_Summary
0,41256141,0.0057,Low Risk,Weight significantly below typical shipments o...
1,20889431,0.0082,Low Risk,Weight significantly below typical shipments o...
2,25403940,0.0423,Low Risk,Weight significantly exceeds typical shipments...
3,43489778,0.0143,Low Risk,No major risk signals detected; routine cleara...
4,94895240,0.0104,Low Risk,No major risk signals detected; routine cleara...
5,81282547,0.2601,Low Risk,Large mismatch between declared and measured w...
6,13935563,0.7547,Critical,Weight significantly below typical shipments o...
7,57224212,0.0445,Low Risk,Shipment behavior detected as anomalous by uns...
8,21579017,0.7771,Critical,Container flagged by the ensemble risk model a...
9,97545517,0.0049,Low Risk,Weight significantly below typical shipments o...
